In [ ]:
"""# Detection → Classification → Evaluation Demo

This notebook demonstrates the baseline pipeline included in this repository:

- Generate synthetic sequences with labeled maneuvers
- Preprocess and extract features
- Run threshold-based detection
- Train a simple classifier on aggregated segment features
- Evaluate detection (IoU, precision/recall) and classification (accuracy, F1)
- Run the CLI from the notebook and show reproducibility steps (also suitable for CI)
"""

## 1. Install and setup

Create a virtual environment (if needed), install dependencies, and assert the package imports correctly.

In [ ]:
import sys, os
# ensure local src is available
sys.path.insert(0, os.path.abspath(os.path.join("..", "..", "src")))

import maneuvers
print('maneuvers version:', getattr(maneuvers, '__version__', 'unknown'))

# quick check of important submodules
import maneuvers.data.loader as loader
import maneuvers.preprocessing as preprocessing
import maneuvers.detection as detection
import maneuvers.classify as classify
import maneuvers.eval as evalmod
print('Submodules loaded OK')

## 2. Generate and save synthetic sequences

Create a few synthetic sequences with varying sampling rates and save them to `examples/data/` for reproducibility.

In [ ]:
from maneuvers.data.loader import generate_synthetic_sequence
import numpy as np
import json
import os

os.makedirs(os.path.join('..','data'), exist_ok=True)

seq1 = generate_synthetic_sequence(duration_s=10.0, fs=100, seed=0)
seq2 = generate_synthetic_sequence(duration_s=6.0, fs=50, seed=1)

# simple saver (CSV + JSON for GT)
def save_seq_as_csv(seq, path_csv, path_gt):
    import csv
    with open(path_csv, 'w', newline='') as fh:
        writer = csv.writer(fh)
        writer.writerow(['t','ax','ay','az','gx','gy','gz'])
        for i,t in enumerate(seq.timestamps):
            ax,ay,az = seq.accel[i]
            gx,gy,gz = seq.gyro[i]
            writer.writerow([f"{t:.6f}", f"{ax:.6f}", f"{ay:.6f}", f"{az:.6f}", f"{gx:.6f}", f"{gy:.6f}", f"{gz:.6f}"])
    with open(path_gt, 'w') as fh:
        json.dump({'segments': seq.segments}, fh)

save_seq_as_csv(seq1, os.path.join('..','data','synthetic_100hz.csv'), os.path.join('..','data','synthetic_100hz_gt.json'))
save_seq_as_csv(seq2, os.path.join('..','data','synthetic_50hz.csv'), os.path.join('..','data','synthetic_50hz_gt.json'))

print('Saved synthetic sequences to examples/data/')

## 3. Visualize sequences and ground-truth maneuvers

Plot accelerometer magnitude and overlay GT segments.

In [ ]:
import matplotlib.pyplot as plt
from maneuvers.preprocessing import compute_features_from_sequence

feats = compute_features_from_sequence(seq1)

def plot_sequence_with_segments(seq, feats, gt_segments, pred_segments=None):
    t = feats['t'].values
    plt.figure(figsize=(12,3))
    plt.plot(t, feats['accel_mag'], label='accel_mag', alpha=0.6)
    plt.plot(t, feats['accel_smooth'], label='accel_smooth', linewidth=2)
    for s,e,label in gt_segments:
        plt.axvspan(seq.timestamps[s], seq.timestamps[e-1], color='green', alpha=0.15, label=f'GT: {label}')
    if pred_segments:
        for s,e in pred_segments:
            plt.axvspan(seq.timestamps[s], seq.timestamps[e-1], color='red', alpha=0.12, label='Pred')
    plt.legend(loc='upper right')
    plt.xlabel('time (s)')
    plt.tight_layout()
    plt.show()

plot_sequence_with_segments(seq1, feats, seq1.segments)

## 4. Baseline threshold detection

Sweep a few thresholds and inspect results.

In [ ]:
from maneuvers.detection import detect_segments

for thr in [0.2, 0.4, 0.6]:
    preds = detect_segments(feats, method='threshold', threshold=thr, min_len=5)
    print(f"thr={thr}: {len(preds)} segments")
    plot_sequence_with_segments(seq1, feats, seq1.segments, preds)

# store one detection result
preds = detect_segments(feats, method='threshold', threshold=0.4, min_len=5)
print('Selected preds:', preds)

## 5. Segment feature extraction and simple classifier

Aggregate per-ground-truth segment features and train a lightweight classifier (LogisticRegression). Save the trained model.

In [ ]:
from sklearn.linear_model import LogisticRegression
from maneuvers.classify import build_training_data_from_sequence, train_classifier, save_model, load_model, predict_segment_labels

X, y = build_training_data_from_sequence(seq1, feats)
print('Training examples:', X.shape, 'labels:', set(y))

# basic classifier (works on aggregated features)
clf = LogisticRegression(max_iter=1000)
clf.fit(X, y)

# predict labels for detected segments (simple aggregated features)
import numpy as np

from maneuvers.classify import segment_aggregated_features

def predict_labels_direct(clf, feats_df, segments):
    Xs = []
    for s,e in segments:
        seg_feat = segment_aggregated_features(feats_df, (s, e))
        Xs.append(seg_feat)
    preds_local = clf.predict(np.vstack(Xs))
    return list(preds_local)

pred_labels = predict_labels_direct(clf, feats, preds)
print('Predicted labels for predicted segments:', list(zip(preds, pred_labels)))

# save model for demo
import joblib
os.makedirs(os.path.join('..','..','examples'), exist_ok=True)
joblib.dump({'model': clf}, os.path.join('..','..','examples','model_demo_lr.joblib'))
print('Saved demo model to examples/model_demo_lr.joblib')

## 6. Evaluate detection and classification

Compute detection metrics (IoU/precision/recall/F1) and classification metrics (accuracy, F1). Plot confusion matrix and PR curve examples.

In [ ]:
from maneuvers.eval import evaluate_detection, segment_iou
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve, roc_auc_score, roc_curve
import matplotlib.pyplot as plt

# detection evaluation
gt = [(s,e) for s,e,_ in seq1.segments]
det_res = evaluate_detection(gt, preds, iou_thresh=0.5)
print('Detection evaluation:', det_res)

# classification evaluation (match predicted -> gt by IoU)
# naive greedy matching
matches = []
used_pred = set()
used_gt = set()
for gi, g in enumerate(gt):
    best_i = -1
    best_iou = 0.0
    for pi, p in enumerate(preds):
        if pi in used_pred:
            continue
        iou = segment_iou(g,p)
        if iou > best_iou:
            best_iou = iou
            best_i = pi
    if best_i >= 0 and best_iou >= 0.5:
        matches.append((gi, best_i))
        used_pred.add(best_i)
        used_gt.add(gi)

true_labels = []
pred_labels_match = []
for gi, pi in matches:
    true_labels.append(seq1.segments[gi][2])
    pred_labels_match.append(pred_labels[pi])

if true_labels:
    print('Classification report (matched segments):')
    print(classification_report(true_labels, pred_labels_match))
    cm = confusion_matrix(true_labels, pred_labels_match, labels=list(set(true_labels+pred_labels_match)))
    plt.figure(figsize=(4,4))
    plt.imshow(cm, cmap='Blues')
    plt.title('Confusion matrix')
    plt.colorbar()
    plt.tight_layout()
    plt.show()
else:
    print('No matched segments to evaluate classification.')

## 7. Run the pipeline via CLI from the notebook

Invoke the package CLI using subprocess and parse outputs programmatically. This mirrors how CI would run the commands.

In [ ]:
import subprocess

print('\nCLI detect-synthetic:')
print(subprocess.run(['python','-m','maneuvers.cli','detect-synthetic','--duration','5','--fs','50','--threshold','0.2'], capture_output=True, text=True).stdout)

print('\nCLI eval-synthetic:')
print(subprocess.run(['python','-m','maneuvers.cli','eval-synthetic','--duration','5','--fs','50','--threshold','0.2'], capture_output=True, text=True).stdout)

## 8. CI workflow snippet (GitHub Actions)

This cell shows the CI snippet used to execute the notebook in CI and upload artifacts. In the repository the workflow has been configured to run the demo and upload an HTML artifact.

In [ ]:
ci_yaml = '''name: CI

on:
  push:
    branches: [ main ]
  pull_request:
    branches: [ main ]

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v4
        with:
          python-version: 3.10
      - run: |
          python -m pip install --upgrade pip
          pip install -r requirements.txt
          pip install -e .
          pip install pytest black flake8 nbval nbconvert
      - run: pytest -q --nbval-lax examples/notebooks/detection_classification_demo.ipynb
      - run: black --check .
      - run: flake8 .
      - run: jupyter nbconvert --execute --to html examples/notebooks/detection_classification_demo.ipynb --ExecutePreprocessor.timeout=600
'''
print(ci_yaml)

## 9. Save outputs and upload artifacts

Serialize detection results and save to `artifacts/` so CI can upload them for inspection.

In [ ]:
os.makedirs(os.path.join('..','..','artifacts'), exist_ok=True)
import json
with open(os.path.join('..','..','artifacts','detection_results.json'), 'w') as fh:
    json.dump({'gt': gt, 'preds': preds}, fh)
print('Wrote artifacts to artifacts/')

## 10. Utilities & next steps

- Add richer features (spectral features, rolling stats) in `maneuvers.preprocessing`.
- Add cross-validation and model serialization routines in `maneuvers.classify` and improved plotting utilities.
- Integrate nbval into CI to ensure notebooks remain executable on PRs.

In [ ]:
# 6. Evaluate detection and classification
from maneuvers.eval import evaluate_detection, segment_iou
from sklearn.metrics import classification_report, confusion_matrix

# detection evaluation
gt = [(s,e) for s,e,_ in seq1.segments]
det_res = evaluate_detection(gt, preds, iou_thresh=0.5)
print('Detection evaluation:', det_res)

# classification evaluation (match predicted -> gt by IoU)
# naive greedy matching
matches = []
used_pred = set()
used_gt = set()
for gi, g in enumerate(gt):
    best_i = -1
    best_iou = 0.0
    for pi, p in enumerate(preds):
        if pi in used_pred:
            continue
        iou = segment_iou(g,p)
        if iou > best_iou:
            best_iou = iou
            best_i = pi
    if best_i >= 0 and best_iou >= 0.5:
        matches.append((gi, best_i))
        used_pred.add(best_i)
        used_gt.add(gi)

true_labels = []
pred_labels_match = []
for gi, pi in matches:
    true_labels.append(seq1.segments[gi][2])
    pred_labels_match.append(pred_labels[pi])

if true_labels:
    print('Classification report (matched segments):')
    print(classification_report(true_labels, pred_labels_match))
else:
    print('No matched segments to evaluate classification.')

# 7. Run CLI commands programmatically
import subprocess

print('\nCLI detect-synthetic:')
print(subprocess.run(['python','-m','maneuvers.cli','detect-synthetic','--duration','5','--fs','50','--threshold','0.2'], capture_output=True, text=True).stdout)

print('\nCLI eval-synthetic:')
print(subprocess.run(['python','-m','maneuvers.cli','eval-synthetic','--duration','5','--fs','50','--threshold','0.2'], capture_output=True, text=True).stdout)

# 8. CI workflow snippet (displayed for convenience)
ci_yaml = '''name: CI

on:
  push:
    branches: [ main ]
  pull_request:
    branches: [ main ]

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v4
        with:
          python-version: 3.10
      - run: |
          python -m pip install --upgrade pip
          pip install -r requirements.txt
          pip install -e .
          pip install pytest black flake8
      - run: pytest -q
      - run: black --check .
      - run: flake8 .
      - run: jupyter nbconvert --execute --to html examples/notebooks/detection_classification_demo.ipynb --ExecutePreprocessor.timeout=600
'''
print('\nCI workflow preview (snippet):\n')
print(ci_yaml)

# 9. Save artifacts
os.makedirs(os.path.join('..','..','artifacts'), exist_ok=True)
import json
with open(os.path.join('..','..','artifacts','detection_results.json'), 'w') as fh:
    json.dump({'gt': gt, 'preds': preds}, fh)
print('Wrote artifacts to artifacts/')
